# Ön Çalışma: Şekil 1, Şekil 2, Tablo 1 ve metindeki sayıları kendin üret

Bu defter başvurudaki ön çalışmanın **her şeklini, tablosunu ve sayısını** baştan üretir. Hücreleri **yukarıdan aşağı, sırayla**
çalıştır (hücreye tıkla → **Shift + Enter**). Her hücrenin üstünde ne yaptığı ve ekranda **ne görmen gerektiği** yazıyor.
Ölçümlerde rastgele sayı üreteci sabitlendiği için (seed = 42) sonuçlar her çalıştırmada aynıdır.

| Başvurudaki öğe | Bu defterdeki adım |
|---|---|
| Tablo 1 (hedef seçimi) | Adım 3 |
| WASP-12 b katalog sonucu | Adım 4 |
| Sektör 20 doğrulaması (−1,7 s; 0,40σ) | Adım 5 |
| Şekil 1 | Adım 6–7 |
| 174 yeni TESS zamanı ve Şekil 2 | Adım 8–9 |

Kullanılan her şey herkese açık: VizieR (Ivshina ve Winn 2022; Wang vd. 2024; Knutson vd. 2014; Bouma vd. 2020; Bonomo vd. 2017),
NASA MAST (TESS ışık eğrileri), NASA Exoplanet Archive, DACE. Süre: yaklaşık 30–40 dakika (çoğu bekleme).

## Adım 1 — Gerekli Python paketlerini kur
`astropy` (zaman/koordinat), `astroquery` (arşivden indirme), `batman` (geçiş modeli), `emcee` (MCMC), `dace-query` (DACE RV arşivi).

In [ ]:
!pip install -q astropy astroquery batman-package emcee dace-query

## Adım 2 — Kod dosyalarını oluştur
Aşağıdaki hücreler kodları Colab klasörüne kaydeder; her biri için yalnızca "Writing ..." görmelisin. Her fonksiyonun ne yaptığı kodun içinde yazıyor.
- `oc_diyagrami.py`: geçiş zamanlarını indirir, epokları hesaplar, hata çubuklarını gerçekçi hale getirir, doğru/parabol uydurur, dP/dt ve ΔBIC'i yazar, O−C şeklini çizer.
- `tess_zaman.py`: TESS ışık eğrisini indirir ve her geçişin orta zamanını ölçer.
- `rv_analiz.py`: radyal hız (RV) verisine yıldız eğilimi uydurur ve Rømer etkisinin öngördüğü dönem değişimini hesaplar.
- `secim_tablosu.py`: Tablo 1'i (14 adaydan hedef seçimi) herkese açık kaynaklardan hesaplar.

In [ ]:
%%writefile oc_diyagrami.py
"""O-C diyagrami: Ivshina & Winn (2022) + Wang vd. (2024) gecis zamanlari.
Kullanim:  python oc_diyagrami.py "WASP-12 b" WASP-012 [tess_WASP-12.csv ...]
"""
import sys, io, urllib.parse, urllib.request
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

VIZIER = "https://vizier.cfa.harvard.edu/viz-bin/asu-tsv"   # Harvard aynasi; olmazsa vizier.cds.unistra.fr
YIL_S = 365.25 * 86400.0                                    # bir yil, saniye
EK_DOSYALAR = []
BELIRSIZ_BJD_DAHIL = True                                   # False yapip sonucun degisip degismedigine bak


def vizier(tablo, **filtre):
    """VizieR tablosunu indir; satir listesi (sozluk) dondur."""
    p = {"-source": tablo, "-out.max": "99999", "-out.all": ""}
    p.update(filtre)
    url = VIZIER + "?" + urllib.parse.urlencode(p)
    metin = urllib.request.urlopen(url, timeout=120).read().decode()
    satir = [s for s in metin.splitlines() if s and not s.startswith("#")]
    if not satir:                                               # eslesen kayit yok
        return []
    bas = [h.strip() for h in satir[0].split("\t")]
    return [dict(zip(bas, [x.strip() for x in s.split("\t")])) for s in satir[3:]]


def zamanlari_topla(gezegen, iw_adi):
    """O: gozlenen orta gecis zamanlari (BJD_TDB), hatalari ve kaynak etiketi."""
    t, s, k = [], [], []
    for r in vizier("J/ApJS/259/62/table4", Sys=iw_adi):        # literatur + I&W'nin TESS zamanlari
        sistem = r["n_TTmid"]
        if sistem not in ("BJD_TDB", "BJD_TT"):                  # TT ile TDB farki < 2 ms: ihmal edilir
            if sistem == "BJD" and BELIRSIZ_BJD_DAHIL:             # "BJD" = UTC mi TDB mi belli degil
                print("UYARI: zaman sistemi belirsiz (BJD), makaleden kontrol et:", r["TTmid"], r["Ref"])
            else:
                print("ATLANDI: zaman sistemi", sistem, r["TTmid"], r["Ref"])
                continue
        t.append(float(r["TTmid"])); s.append(float(r["e_TTmid"]))
        k.append("TESS" if r["Ref"] == "This work" else "Literatur")
    for r in vizier("J/ApJS/270/14/table1", Name=gezegen):      # Wang vd. TESS zamanlari
        tc = float(r["Tc"])
        if np.min(np.abs(np.array(t) - tc)) < 0.01:              # ayni gecis zaten var: tekrar sayma
            continue
        t.append(tc); s.append(float(r["e_Tc"])); k.append("TESS")
    for dosya in EK_DOSYALAR:                                     # kendi olctugun TESS zamanlari
        for _, tc, h in np.atleast_2d(np.loadtxt(dosya, delimiter=",")):
            if np.min(np.abs(np.array(t) - tc)) < 0.01:
                continue
            t.append(tc); s.append(h); k.append("TESS (bu proje)")
    i = np.argsort(t)
    return np.array(t)[i], np.array(s)[i], np.array(k)[i]


def uydur(E, t, s, derece):
    """Agirlikli en kucuk kareler. derece=1 dogrusal, 2 kuadratik. (katsayilar, kovaryans, chi2)."""
    X = np.vander(E, derece + 1, increasing=True)              # [1, E, E^2]
    A, b = X / s[:, None], t / s
    p, *_ = np.linalg.lstsq(A, b, rcond=None)
    kov = np.linalg.inv(A.T @ A)
    chi2 = float(np.sum((b - A @ p) ** 2))
    return p, kov, chi2


def analiz(E, t, s):
    """Dogrusal vs kuadratik: dP/dt (ms/yil), hatasi, Delta BIC."""
    n = len(t)
    p1, _, c1 = uydur(E, t, s, 1)
    p2, k2, c2 = uydur(E, t, s, 2)
    P = p2[1]
    pdot = 2 * p2[2] / P * YIL_S * 1e3                          # dP/dt = (1/P) dP/dN, dP/dN = 2c
    pdot_h = 2 * np.sqrt(k2[2, 2]) / P * YIL_S * 1e3
    dbic = (c1 + 2 * np.log(n)) - (c2 + 3 * np.log(n))
    return dict(p1=p1, p2=p2, pdot=pdot, pdot_h=pdot_h, dbic=dbic, chi2_1=c1, chi2_2=c2, n=n)


def sigma_kirp(E, t, s, n_sigma=5):
    """Wang vd. gibi: kuadratik artiklarda n-sigma disindakileri yinelemeli at."""
    tut = np.ones(len(t), bool)
    while True:
        p, _, _ = uydur(E[tut], t[tut], s[tut], 2)
        r = t - np.polyval(p[::-1], E)
        m, sd = r[tut].mean(), r[tut].std()
        yeni = np.abs(r - m) < n_sigma * sd
        if (yeni == tut).all():
            return tut
        tut = yeni


def hata_olcekle(E, t, s, k, tur=5):
    """Hata cubuklarini veri grubu (literatur / TESS) basina, kuadratik model artiklarinin
    indirgenmis ki-karesi 1 olacak sekilde buyutur (hicbir zaman kucultmez)."""
    s = s.copy()
    gruplar = [k == "Literatur", k != "Literatur"]
    for _ in range(tur):
        p, _, _ = uydur(E, t, s, 2)
        r = (t - np.polyval(p[::-1], E)) / s
        for g in gruplar:
            if g.sum() > 3:
                s[g] *= np.sqrt(max(np.sum(r[g] ** 2) / (g.sum() - 1), 1.0))
    return s


def loocv(E, t, s):
    """Her noktayi tek tek cikar, yeniden uydur; 3-sigma ve DBIC>10 kosulu hep saglaniyor mu?"""
    sonuc = []
    for i in range(len(t)):
        m = np.arange(len(t)) != i
        a = analiz(E[m], t[m], s[m])
        sonuc.append(abs(a["pdot"]) / a["pdot_h"] >= 3 and a["dbic"] > 10)
    return np.array(sonuc)


if __name__ == "__main__":
    gezegen, iw_adi, EK_DOSYALAR = sys.argv[1], sys.argv[2], sys.argv[3:]
    t, s, k = zamanlari_topla(gezegen, iw_adi)
    # C icin baslangic efemerisi: Wang Tablo 2 (yoksa NASA Exoplanet Archive'dan T0, P)
    ef = [r for r in vizier("J/ApJS/270/14/table2", Name=gezegen)][0]
    T0, P = float(ef["T0"]), float(ef["P0"])
    E = np.round((t - T0) / P)
    E = E - np.round(np.median(E))                              # epoklari ortala: sayisal kararlilik
    tut = sigma_kirp(E, t, s, 5)
    E, t, s, k = E[tut], t[tut], s[tut], k[tut]
    s = hata_olcekle(E, t, s, k)                               # gercekci hata cubuklari
    a = analiz(E, t, s)
    gecti = loocv(E, t, s)
    print(f"{gezegen}: n={a['n']} (TESS {np.sum(k!='Literatur')}), dP/dt = {a['pdot']:.2f} +- {a['pdot_h']:.2f} ms/yil, "
          f"DBIC = {a['dbic']:.1f}, LOOCV: {np.sum(~gecti)} noktada kosul bozuluyor -> {'GECTI' if gecti.all() else 'KALDI'}")

    # O-C: dogrusal efemerise gore (C = T0 + P*E)
    C = a["p1"][0] + a["p1"][1] * E
    oc_dk = (t - C) * 1440
    ee = np.linspace(E.min(), E.max(), 400)
    kuad = (np.polyval(a["p2"][::-1], ee) - (a["p1"][0] + a["p1"][1] * ee)) * 1440
    fig, ax = plt.subplots(figsize=(7, 4))
    for etiket, renk in [("Literatur", "tab:orange"), ("TESS", "tab:green"), ("TESS (bu proje)", "tab:red")]:
        m = k == etiket
        if not m.any():
            continue
        ax.errorbar(E[m], oc_dk[m], s[m] * 1440, fmt="o", ms=3, color=renk, label=etiket, alpha=.8)
    ax.plot(ee, kuad, "b-", label=f"kuadratik: dP/dt = {a['pdot']:.1f} ± {a['pdot_h']:.1f} ms/yıl")
    ax.axhline(0, color="k", lw=.8, ls="--", label="doğrusal (sabit dönem)")
    ax.set_xlabel("Epok (E)"); ax.set_ylabel("O − C (dakika)")
    ax.set_title(f"{gezegen}  (ΔBIC = {a['dbic']:.1f}, n = {a['n']})")
    ax.legend(fontsize=8); fig.tight_layout()
    ad = gezegen.replace(" ", "")
    fig.savefig(f"OC_{ad}.png", dpi=200)
    np.savetxt(f"zamanlar_{ad}.csv", np.column_stack([E, t, s, oc_dk]), delimiter=",",
               header="E,T_gozlenen_BJD_TDB,hata_gun,OC_dakika", fmt=["%d", "%.6f", "%.6f", "%.3f"])


In [ ]:
%%writefile tess_zaman.py
"""TESS isik egrisinden orta gecis zamanlari (O) olcer.
Kullanim:  python tess_zaman.py WASP-12 97.6366450 29.6722662 1.09141890 2457607.51930 71 72
           (hedef, RA, Dec, P [gun], T0 [BJD_TDB], sektor listesi)
"""
import sys
import numpy as np
import batman, emcee
from scipy.optimize import least_squares
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astroquery.mast import Observations

BTJD0 = 2457000.0                         # TESS zamani: BTJD = BJD_TDB - 2457000


def indir(ra, dec, sektorler):
    """MAST'tan SPOC 120 s isik egrisi (*_lc.fits) dosyalarini indirir."""
    obs = Observations.query_criteria(coordinates=SkyCoord(ra, dec, unit="deg"), radius="0.003 deg",
                                      obs_collection="TESS", dataproduct_type="timeseries",
                                      provenance_name="SPOC")
    obs = obs[np.isin(obs["sequence_number"], sektorler)]
    urun = Observations.get_product_list(obs)
    urun = urun[[str(f).endswith("_lc.fits") for f in urun["productFilename"]]]
    liste = Observations.download_products(urun, download_dir="veri_ham")
    return sorted(str(p) for p in liste["Local Path"])        # yalnizca BU hedefin dosyalari


def oku(dosya):
    """TIME (BTJD), PDCSAP_FLUX; kalite bayragi 0 olmayan ve NaN noktalari atar, 1'e normalize eder."""
    d = fits.getdata(dosya, 1)
    m = (d["QUALITY"] == 0) & np.isfinite(d["TIME"]) & np.isfinite(d["PDCSAP_FLUX"])
    t, f, e = d["TIME"][m], d["PDCSAP_FLUX"][m], d["PDCSAP_FLUX_ERR"][m]
    return t + BTJD0, f / np.nanmedian(f), e / np.nanmedian(f)


def model(t, tc, P, rp, a, inc, u1=0.4, u2=0.2):
    p = batman.TransitParams()
    p.t0, p.per, p.rp, p.a, p.inc, p.ecc, p.w = tc, P, rp, a, inc, 0.0, 90.0
    p.u, p.limb_dark = [u1, u2], "quadratic"
    return batman.TransitModel(p, t).light_curve(p)


def efemeris_merkezle(t, f, T0, P):
    """Eski bir T0 TESS'e kadar dakikalarca kaymis olabilir: katlanmis isik egrisinde
    en derin noktayi kutu taramasiyla bul ve T0'i oraya tasi (BLS benzeri basit tarama)."""
    n_ort = np.round((np.median(t) - T0) / P)
    T0 = T0 + n_ort * P                                       # T0'i TESS verisinin ortasina tasi
    kaymalar = np.arange(-0.15, 0.15, 0.001)
    derinlik = []
    for dt in kaymalar:
        x = ((t - T0 - dt) / P + 0.5) % 1 - 0.5
        derinlik.append(np.mean(f[np.abs(x * P) < 0.03]))     # +-43 dk kutu
    return T0 + kaymalar[int(np.argmin(derinlik))]


def sekil_uydur(t, f, e, T0, P):
    """Tum gecisleri katlayip (phase-fold) gecis seklini (rp, a, inc) bir kez belirler."""
    faz = ((t - T0) / P + 0.5) % 1 - 0.5                     # -0.5..0.5, gecis 0'da
    m = np.abs(faz) < 0.1
    x = faz[m] * P
    r = least_squares(lambda q: (f[m] - q[3] * model(x, 0.0, P, *q[:3])) / e[m],
                      [0.1, 4.0, 85.0, 1.0], bounds=([0.01, 1.5, 70, 0.9], [0.3, 20, 90, 1.1]))
    return r.x[:3]


def tek_gecis(t, f, e, tc0, P, sekil, mcmc=True):
    """Bir gecis: serbest parametreler tc + dogrusal taban (c0, c1). Sonuc: tc, hata (gun)."""
    def m(q):
        return model(t, q[0], P, *sekil) * (q[1] + q[2] * (t - tc0))
    ls = least_squares(lambda q: (f - m(q)) / e, [tc0, 1.0, 0.0])
    if not mcmc:
        J = ls.jac; kov = np.linalg.inv(J.T @ J)
        return ls.x[0], np.sqrt(kov[0, 0]), f - m(ls.x)
    def lnp(q):
        if abs(q[0] - tc0) > 0.05:
            return -np.inf
        return -0.5 * np.sum(((f - m(q)) / e) ** 2)
    np.random.seed(42)                                        # sabit tohum: her calistirmada ayni sonuc (yeniden uretilebilirlik)
    sm = emcee.EnsembleSampler(12, 3, lnp)
    sm.run_mcmc(ls.x + 1e-5 * np.random.randn(12, 3), 2000, progress=False)
    z = sm.get_chain(discard=500, flat=True)[:, 0]
    q16, q50, q84 = np.percentile(z, [16, 50, 84])
    return q50, 0.5 * (q84 - q16), f - m(ls.x)


if __name__ == "__main__":
    hedef, ra, dec, P, T0 = sys.argv[1], *map(float, sys.argv[2:6])
    sektorler = [int(x) for x in sys.argv[6:]]
    t_hep, f_hep, e_hep = [], [], []
    for d in indir(ra, dec, sektorler):
        t, f, e = oku(d); t_hep.append(t); f_hep.append(f); e_hep.append(e)
    t, f, e = map(np.concatenate, (t_hep, f_hep, e_hep))
    T0_eski = T0
    T0 = efemeris_merkezle(t, f, T0, P)
    for tur in (1, 2):                                        # 1. tur: hizli (en kucuk kareler); 2. tur: MCMC
        sekil = sekil_uydur(t, f, e, T0, P)
        print(f"Tur {tur}: Rp/R* = {sekil[0]:.4f}, a/R* = {sekil[1]:.2f}, i = {sekil[2]:.2f}")
        sure = P / np.pi / sekil[1]                          # kabaca gecis suresi (gun)
        satir = []
        for n in range(int(np.ceil((t.min() - T0) / P)), int(np.floor((t.max() - T0) / P)) + 1):
            tc0 = T0 + n * P
            w = np.abs(t - tc0) < 1.5 * sure                   # gecisin iki yani + taban
            giris = np.any((t > tc0 - 0.6 * sure) & (t < tc0 - 0.4 * sure))
            cikis = np.any((t > tc0 + 0.4 * sure) & (t < tc0 + 0.6 * sure))
            if w.sum() < 50 or not (giris and cikis):           # kismi gecis: kullanma
                continue
            tc, h, art = tek_gecis(t[w], f[w], e[w], tc0, P, sekil, mcmc=(tur == 2))
            satir.append((n, tc, h))
        n_, tc_, h_ = np.array(satir).T
        q = np.polyfit(n_, tc_, 1, w=1 / h_)                  # TESS zamanlarindan yerel efemeris
        P, T0 = q[0], q[1]
    for n, tc, h in satir:
        print(f"Tc = {tc:.6f} BJD_TDB  +- {h * 86400:5.1f} s")
    satir = [(np.round((tc - T0_eski) / P), tc, h) for _, tc, h in satir]   # epok: girilen T0'a gore
    np.savetxt(f"tess_{hedef}.csv", satir, delimiter=",", fmt=["%d", "%.6f", "%.6f"],
               header="epok,Tc_BJD_TDB,hata_gun")


In [ ]:
%%writefile rv_analiz.py
"""Radyal hiz (RV) egilimi -> Romer etkisinin ongordugu donem degisimi.
Kullanim:
  python rv_analiz.py knutson HAT-P-7 2.204737 2454954.358572 18.28 4.05
  python rv_analiz.py bouma   WASP-4  1.338231 2455804.515679 -6.43 0.56
  python rv_analiz.py dosya.csv  AD   P  T0  dPdt  hata      (csv: BJD,RV,hata,alet)
Son iki sayi: O-C'den buldugun dP/dt ve hatasi (ms/yil).
"""
import sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from oc_diyagrami import vizier

C_ISIK = 2.998e8                       # m/s
YIL_S = 365.25 * 86400.0


def rv_oku(kaynak, yildiz):
    """(t [BJD], v [m/s], hata [m/s], alet) dizileri."""
    if kaynak == "knutson":                                     # Keck/HIRES, Knutson vd. 2014
        r = vizier("J/ApJ/785/126/table3", Name=yildiz)
        t = [float(x["BJD"]) for x in r]; v = [float(x["RV"]) for x in r]
        s = [float(x["e_RV"]) for x in r]; a = ["HIRES"] * len(r)
    elif kaynak == "bouma":                                     # WASP-4: CORALIE + HARPS + HIRES, Bouma vd. 2020
        r = vizier("J/ApJ/893/L29/table2")
        t = [float(x["BJD"]) + 2450000 for x in r]; v = [float(x["RV"]) for x in r]
        s = [float(x["e_RV"]) for x in r]; a = [x["Tel"] for x in r]
    else:                                                       # kendi derledigin CSV
        d = np.genfromtxt(kaynak, delimiter=",", dtype=None, encoding=None, names=True)
        t, v, s, a = d[d.dtype.names[0]], d[d.dtype.names[1]], d[d.dtype.names[2]], d[d.dtype.names[3]]
    return np.array(t, float), np.array(v, float), np.array(s, float), np.array(a, str)


def rv_uydur(t, v, s, alet, P, T0, egrilik=False):
    """v = gamma_alet + gdot*(t-tref) [+ gddot*(t-tref)^2] + A*sin(faz) + B*cos(faz)  (dairesel yorunge).
    Her alet kendi sifir noktasini (gamma) alir: farkli spektrograflarin RV'leri dogrudan karsilastirilamaz."""
    tref = np.mean(t)
    faz = 2 * np.pi * (t - T0) / P
    aletler = sorted(set(alet))
    kol = [(alet == x).astype(float) for x in aletler]          # alet basina bir gamma
    kol += [(t - tref) / 365.25, np.sin(faz), np.cos(faz)]
    if egrilik:
        kol.append(((t - tref) / 365.25) ** 2)
    X = np.column_stack(kol)
    A, b = X / s[:, None], v / s
    p, *_ = np.linalg.lstsq(A, b, rcond=None)
    kov = np.linalg.inv(A.T @ A)
    chi2 = float(np.sum((b - A @ p) ** 2))
    olcek = np.sqrt(max(chi2 / (len(t) - len(p)), 1.0))      # hatalar kucukse olcekle (yildiz titresimi/jitter)
    return p, np.sqrt(np.diag(kov)) * olcek, X, tref, len(aletler)


def romer_pdot(P, gdot):
    """dP/dt (ms/yil) = P * (dv/dt) / c ;  gdot m/s/yil."""
    return P * 86400 * (gdot / YIL_S) / C_ISIK * YIL_S * 1e3


if __name__ == "__main__":
    kaynak, yildiz = sys.argv[1], sys.argv[2]
    P, T0, pdot_g, pdot_gh = map(float, sys.argv[3:7])
    t, v, s, alet = rv_oku(kaynak, yildiz)
    p, h, X, tref, na = rv_uydur(t, v, s, alet, P, T0)
    gdot, gdot_h = p[na], h[na]                               # m/s/yil
    pr, prh = romer_pdot(P, gdot), romer_pdot(P, gdot_h)
    fark = (pdot_g - pr) / np.hypot(pdot_gh, prh)
    print(f"{yildiz}: {len(t)} RV ({', '.join(sorted(set(alet)))}), {(t.max()-t.min())/365.25:.1f} yil; "
          f"K = {np.hypot(p[na+1], p[na+2]):.1f} m/s")
    print(f"  RV egilimi  dv/dt = {gdot:.2f} +- {gdot_h:.2f} m/s/yil")
    print(f"  Romer'in ongordugu dP/dt = {pr:.2f} +- {prh:.2f} ms/yil;  O-C'den gozlenen {pdot_g} +- {pdot_gh}")
    print(f"  fark = {fark:.1f} sigma ->", "Romer TEK BASINA ACIKLAYABILIR" if abs(fark) < 2 else "Romer TEK BASINA ACIKLAYAMAZ")
    artik = v - X[:, :na] @ p[:na] - X[:, na+1:na+3] @ p[na+1:na+3]   # alet sifirlari ve gezegen sinyali cikarilmis
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for x in sorted(set(alet)):
        m = alet == x
        ax.errorbar(t[m] - 2450000, artik[m], s[m], fmt="o", ms=3, label=x)
    tt = np.linspace(t.min(), t.max(), 100)
    ax.plot(tt - 2450000, gdot * (tt - tref) / 365.25, "r-", label=f"eğilim: {gdot:.1f} ± {gdot_h:.1f} m/s/yıl")
    ax.set_xlabel("BJD − 2450000"); ax.set_ylabel("RV − gezegen − sıfır nok. (m/s)"); ax.set_title(yildiz)
    ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(f"RV_{yildiz}.png", dpi=200)


In [ ]:
%%writefile secim_tablosu.py
"""Tablo 1 (hedef secimi) - tum sutunlar herkese acik kaynaklardan hesaplanir.
Kullanim:  python secim_tablosu.py        (DACE icin: pip install dace-query)
K1: Wang vd. (2024) Tablo 3, LOOCV sutunu "pass" olan 14 aday (makalenin kendisinden; VizieR'de bu sutun yok).
K2: Wang vd. TESS gecis zamani sayisi >= 30   (VizieR J/ApJS/270/14/table1'de satir sayilir)
K3: arsiv RV'sinden Romer ongorusunun belirsizligi < Wang vd. |dP/dt| / 3   (K2'yi gecenler icin)
"""
import csv, io, urllib.parse, urllib.request, collections
import numpy as np
from oc_diyagrami import vizier
from rv_analiz import rv_uydur, romer_pdot

K1 = ["CoRoT-2 b", "HAT-P-37 b", "KELT-16 b", "TrES-1 b", "TrES-5 b", "WASP-4 b", "WASP-10 b", "WASP-12 b",
      "WASP-22 b", "WASP-45 b", "XO-4 b", "HAT-P-7 b", "WASP-1 b", "WASP-46 b"]


def nasa(gezegen):
    q = f"select pl_orbper,pl_tranmid from pscomppars where pl_name='{gezegen}'"
    url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?" + urllib.parse.urlencode({"query": q, "format": "csv"})
    r = list(csv.DictReader(io.StringIO(urllib.request.urlopen(url, timeout=60).read().decode())))[0]
    return float(r["pl_orbper"]), float(r["pl_tranmid"])


def rv_setleri(yildiz, P, T0):
    """Her arsivden (t, v, s, alet) dondurur. Gecis sirasindaki RV'ler ve <3 olcumlu aletler atilir."""
    setler = {}
    r = vizier("J/ApJ/785/126/table3", Name=yildiz)                                  # Knutson vd. 2014
    if r:
        setler["Knutson 2014"] = ([float(x["BJD"]) for x in r], [float(x["RV"]) for x in r],
                                  [float(x["e_RV"]) for x in r], ["HIRES"] * len(r))
    if yildiz == "WASP-4":                                                           # Bouma vd. 2020
        r = vizier("J/ApJ/893/L29/table2")
        setler["Bouma 2020"] = ([float(x["BJD"]) + 2450000 for x in r], [float(x["RV"]) for x in r],
                                [float(x["e_RV"]) for x in r], [x["Tel"] for x in r])
    r = [x for x in vizier("J/A+A/602/A107/table1") if x["Star"].strip() == yildiz]  # Bonomo vd. 2017 (km/s)
    if r:
        setler["Bonomo 2017"] = ([float(x["BJD"]) + 2450000 for x in r], [float(x["RV"]) * 1e3 for x in r],
                                 [float(x["e_RV"]) * 1e3 for x in r], ["HARPS-N"] * len(r))
    try:                                                                             # DACE
        from dace_query.spectroscopy import Spectroscopy
        d = Spectroscopy.get_timeseries(yildiz, sorted_by_instrument=False, output_format="dict")
        if d and "rjd" in d:
            setler["DACE"] = (list(np.array(d["rjd"], float) + 2400000), list(d["rv"]), list(d["rv_err"]),
                              [str(x) for x in d["instrument_name"]])
    except Exception as e:
        print("  DACE okunamadi:", yildiz, type(e).__name__)
    temiz = {}
    for ad, (t, v, s, a) in setler.items():
        t, v, s, a = np.array(t, float), np.array(v, float), np.array(s, float), np.array(a, str)
        faz = ((t - T0) / P + 0.5) % 1 - 0.5
        m = np.isfinite(v) & np.isfinite(s) & (s > 0) & (np.abs(faz) > 0.04)
        u, n = np.unique(a[m], return_counts=True); m &= np.isin(a, u[n >= 3])
        if m.sum() >= 6:
            temiz[ad] = (t[m], v[m], s[m], a[m])
    return temiz


if __name__ == "__main__":
    tess = collections.Counter(r["Name"].strip() for r in vizier("J/ApJS/270/14/table1"))
    t2 = {r["Name"].strip(): r for r in vizier("J/ApJS/270/14/table2") if r["sigma"].strip() == "5"}
    satirlar = []
    for g in K1:
        pd, pe = abs(float(t2[g]["dP/dt"])), float(t2[g]["e_dP/dt"])
        n = tess[g]; sr, kaynak, sonuc = None, "", ""
        if n < 30:
            sonuc = "K2 saglanmiyor"
        else:
            P, T0 = nasa(g)
            enk = {}
            for ad, (t, v, s, a) in rv_setleri(g[:-2], P, T0).items():
                p, h, _, _, na = rv_uydur(t, v, s, a, P, T0)
                enk[ad] = abs(romer_pdot(P, h[na]))
            if enk:
                kaynak = min(enk, key=enk.get); sr = enk[kaynak]
            sonuc = "SECILDI" if (sr is not None and sr < pd / 3) else "K3 saglanmiyor"
        satirlar.append((g, pd / pe, n, sr, kaynak, pd / 3, sonuc))
        print(f"{g:11s} |dP/dt|/sigma={pd/pe:5.1f}  TESS={n:3d}  sigma_Romer={'-' if sr is None else f'{sr:.1f}':>5} {kaynak:12s}"
              f" |dP/dt|/3={pd/3:5.1f}  -> {sonuc}")
    with open("tablo1_secim.csv", "w") as f:
        f.write("aday,dPdt_anlamlilik,TESS_sayisi,sigma_Romer_msyil,RV_kaynagi,Wang_dPdt_bolu_3,sonuc\n")
        for g, sn, n, sr, k, e, s in satirlar:
            f.write(f"{g},{sn:.1f},{n},{'' if sr is None else f'{sr:.2f}'},{k},{e:.1f},{s}\n")


## Adım 3 — Tablo 1: 14 adaydan hedef seçimi
- **K1:** Wang vd. (2024) Tablo 3'te LOOCV sütunu "pass" olan 14 aday (bu sütun makalenin kendisinde; gösterirken makalenin Tablo 3'ünü aç).
- **K2:** Wang vd.'nin TESS geçiş zamanı sayısı ≥ 30 (VizieR'deki tablolarında satırlar sayılır).
- **K3:** Arşiv RV'sinden Rømer öngörüsünün belirsizliği < Wang vd.'nin |dP/dt| değerinin üçte biri (her aday için tüm arşivler denenir, en hassası alınır).

**Görmen gereken (≈3–5 dk):** `SECILDI` yalnızca **TrES-1 b** (σ 2,2 · Bonomo 2017), **WASP-4 b** (0,4 · Bouma 2020), **WASP-12 b** (1,6 · Knutson 2014),
**HAT-P-7 b** (1,2 · Knutson 2014) satırlarında. KELT-16, TrES-5 ve WASP-46 → `K3 saglanmiyor`; diğer yedisi → `K2 saglanmiyor`.
Sonuç `tablo1_secim.csv` dosyasına da yazılır.

In [ ]:
!python secim_tablosu.py

## Adım 4 — Sadece hazır katalog verisiyle WASP-12 b
**Görmen gereken:** `WASP-12 b: n=189 (TESS 82), dP/dt = -28.69 +- 1.03 ms/yil, DBIC = 770.5`
(Literatür değeri −29 ± 2 ms/yıl; Yee vd. 2020.)

In [ ]:
!python oc_diyagrami.py "WASP-12 b" WASP-012

## Adım 5 — Ölçüm yöntemini doğrula: Wang vd.'nin ölçtüğü sektör 20
Wang vd.'nin **zaten ölçtüğü** TESS sektör 20'yi kendimiz ölçüp onların sonucuyla karşılaştırıyoruz (≈3–5 dk).

**Görmen gereken:** `Tur 2: Rp/R* = 0.1138, a/R* = 3.11` ve 21 satır `Tc = ...`

In [ ]:
!python tess_zaman.py WASP-12 97.6366450 29.6722662 1.091418901 2457607.519305 20
!mv tess_WASP-12.csv dogrulama_sektor20.csv

**Görmen gereken:** `olculen 21, ortak 20 | ortalama fark -1.7 s | ortanca |fark|/sigma 0.40 | ortanca hata 45 s`
Sistematik kayma yok, farklar hata payının içinde → ölçüm yöntemi doğrulandı.

In [ ]:
import numpy as np, oc_diyagrami as oc
ben = np.loadtxt("dogrulama_sektor20.csv", delimiter=",")
w = oc.vizier("J/ApJS/270/14/table1", Name="WASP-12 b")
wt = np.array([float(x["Tc"]) for x in w]); ws = np.array([float(x["e_Tc"]) for x in w])
fark, sig = [], []
for _, tc, h in ben:
    j = np.argmin(abs(wt - tc))
    if abs(wt[j] - tc) < 0.02:
        fark.append((tc - wt[j]) * 86400); sig.append((tc - wt[j]) / np.hypot(h, ws[j]))
fark, sig = np.array(fark), np.array(sig)
print(f"olculen {len(ben)}, ortak {len(fark)} | ortalama fark {fark.mean():+.1f} s | ortanca |fark|/sigma {np.median(abs(sig)):.2f} | ortanca hata {np.median(ben[:, 2]) * 86400:.0f} s")

## Adım 6 — Wang vd. sonrası yeni TESS sektörleri (71 ve 72)
**Görmen gereken (≈5–8 dk):** `Tur 2: Rp/R* = 0.1131, a/R* = 3.12` ve 38 satır `Tc = ...`

In [ ]:
!python tess_zaman.py WASP-12 97.6366450 29.6722662 1.091418901 2457607.519305 71 72

## Adım 7 — Şekil 1
**Görmen gereken:** `WASP-12 b: n=227 (TESS 120), dP/dt = -30.90 +- 0.90 ms/yil, DBIC = 1177.1`
`OC_WASP-12b.png` dosyası **başvurudaki Şekil 1'dir**.

In [ ]:
!python oc_diyagrami.py "WASP-12 b" WASP-012 tess_WASP-12.csv
from IPython.display import Image
Image("OC_WASP-12b.png")

## Adım 8 — Diğer üç hedefin yeni TESS sektörleri (≈15 dk)
**Görmen gereken:**
- `WASP-4 b: n=197 (TESS 122), dP/dt = -4.79 +- 0.40 ms/yil, DBIC = 137.7`
- `HAT-P-7 b: n=117 (TESS 106), dP/dt = 30.00 +- 4.73 ms/yil, DBIC = 35.5`
- `TrES-1 b: n=94 (TESS 55), dP/dt = -4.26 +- 2.63 ms/yil, DBIC = -1.9` (anlamlı değişim yok)

In [ ]:
!python tess_zaman.py WASP-4 353.5629149 -42.0617794 1.338231 2455804.515679 36 69 96 103 105 106
!python tess_zaman.py HAT-P-7 292.2471900 47.9695429 2.204737 2454954.358572 74 75 78 81 82 86
!python tess_zaman.py TrES-1 286.0408755 36.6325360 3.03007 2453186.806341 74 78 80 81 86
!python oc_diyagrami.py "WASP-4 b" WASP-004 tess_WASP-4.csv
!python oc_diyagrami.py "HAT-P-7 b" HAT-P-07 tess_HAT-P-7.csv
!python oc_diyagrami.py "TrES-1 b" TrES-1 tess_TrES-1.csv

**Görmen gereken:** `WASP-12 38 | WASP-4 75 | HAT-P-7 40 | TrES-1 21 | TOPLAM 174` (metindeki "174 yeni TESS geçiş zamanı").

In [ ]:
import numpy as np
say = {h: len(np.loadtxt(f"tess_{h}.csv", delimiter=",")) for h in ["WASP-12", "WASP-4", "HAT-P-7", "TrES-1"]}
print(" | ".join(f"{h} {n}" for h, n in say.items()), "| TOPLAM", sum(say.values()))

## Adım 9 — Şekil 2: dört paneli birleştir
`sekil2_dort_hedef_OC.png` **başvurudaki Şekil 2'dir.** Sol menüdeki klasör simgesinden PNG ve CSV dosyalarını indir.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as im
fig, ax = plt.subplots(2, 2, figsize=(14, 8.2))
for a, ad in zip(ax.flat, ["WASP-12b", "WASP-4b", "HAT-P-7b", "TrES-1b"]):
    a.imshow(im.imread(f"OC_{ad}.png")); a.axis("off")
plt.tight_layout(); plt.savefig("sekil2_dort_hedef_OC.png", dpi=150); plt.show()